# RQ1: Impact of Chunking Strategy on Retrieval Quality

This notebook evaluates fixed-size vs structure-aware chunking strategies on retrieval quality using the evaluation harness against the Golden Standard.

**Research Question:** Does structure-aware chunking improve retrieval performance compared to fixed-size chunking?

**Evaluation Method:**
- Metric-driven approach comparing recall, precision, MRR, coverage, and answer quality metrics
- Benchmark questions: Q1, Q2, Q3, Q8, Q12 (5 questions across 17 companies)
- Golden Standard provides reference answers and page citations

**Inputs:** Golden Standard, pre-built indices
**Outputs:** RQ1 evaluation results saved to `data/artifacts/evaluation_results/rq1_results.json`

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Any

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.config import load_golden_standard, get_default_pipeline_config, BENCHMARK_QUESTION_IDS, ARTIFACTS_DIR
from rag.pipeline import build_index, load_pipeline, run_query
from rag.evaluator import evaluate_pipeline_config

print("Imports successful. Project root:", project_root)

In [ ]:
# Enable smoke test mode for faster iteration
# Set to False for full evaluation on all benchmark questions and companies
SMOKE_TEST = True

if SMOKE_TEST:
    print("[SMOKE TEST MODE] Using subset: 1 company, 1 question")
    test_companies = ["AkerBP_2024"]
    test_question_ids = ["Q1"]
else:
    print("[FULL EVALUATION] Using all benchmark questions and companies")
    test_companies = None  # None means all companies
    test_question_ids = BENCHMARK_QUESTION_IDS

In [ ]:
# Load the Golden Standard evaluation dataset
golden_standard = load_golden_standard()
print(f"Loaded Golden Standard: {len(golden_standard)} questions, {len(golden_standard.get('companies', []))} companies")
print(f"Sample question keys: {list(golden_standard.keys())[:5]}")

In [ ]:
# Define RQ1 configurations: compare chunking strategies
default_config = get_default_pipeline_config()

rq1_configs = {
    "RQ1_fixed_512": {
        **default_config,
        "chunking_strategy": "fixed_size",
        "chunk_size_tokens": 512,
        "chunk_overlap_tokens": 64,
    },
    "RQ1_structure_aware": {
        **default_config,
        "chunking_strategy": "structure_aware",
        "chunk_size_tokens": 512,
        "chunk_overlap_tokens": 64,
    },
}

print("RQ1 Configurations:")
for name, config in rq1_configs.items():
    print(f"  {name}:")
    print(f"    Chunking: {config['chunking_strategy']}")
    print(f"    Retrieval: {config['retrieval_method']}")
    print(f"    Top-K: {config['top_k']}")

## Run RQ1 Evaluation

In [ ]:
# Run evaluation for each RQ1 configuration
rq1_results = {}

for config_name, config in rq1_configs.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {config_name}")
    print(f"{'='*60}")
    
    # Build index for this configuration
    print(f"Building index with {config['chunking_strategy']} chunking...")
    vector_index, bm25_index, chunks = build_index(
        config=config,
        force_rebuild=False,
        companies=test_companies
    )
    print(f"Index built: {len(chunks)} chunks")
    
    # Evaluate this configuration
    print(f"Running evaluation...")
    result = evaluate_pipeline_config(
        config_name=config_name,
        retriever=None,  # Will be created internally
        config=config,
        golden_standard=golden_standard,
        question_ids=test_question_ids,
        companies=test_companies,
        skip_llm_judge=SMOKE_TEST,
        skip_bertscore=SMOKE_TEST
    )
    
    rq1_results[config_name] = result
    print(f"Evaluation complete for {config_name}")

## RQ1 Results Comparison

In [ ]:
# Extract aggregate metrics from results
metrics_data = []

for config_name, result in rq1_results.items():
    agg = result.get("aggregate", {})
    
    metrics_data.append({
        "Configuration": config_name,
        "Recall@5": agg.get("retrieval_recall_at_5", 0),
        "Recall@10": agg.get("retrieval_recall_at_10", 0),
        "Recall@15": agg.get("retrieval_recall_at_15", 0),
        "Precision@5": agg.get("retrieval_precision_at_5", 0),
        "Precision@10": agg.get("retrieval_precision_at_10", 0),
        "Precision@15": agg.get("retrieval_precision_at_15", 0),
        "MRR": agg.get("retrieval_mrr", 0),
        "Coverage": agg.get("retrieval_coverage", 0),
        "Token F1": agg.get("answer_token_f1", 0),
        "Citation F1": agg.get("answer_citation_f1", 0),
    })

comparison_df = pd.DataFrame(metrics_data)
print("\nRQ1 Metrics Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Create visualization comparing retrieval quality
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('RQ1: Chunking Strategy Comparison', fontsize=14, fontweight='bold')

configs = list(rq1_results.keys())
x_pos = np.arange(len(configs))
width = 0.35

# Recall metrics
ax = axes[0, 0]
recalls = [rq1_results[c].get("aggregate", {}).get(f"retrieval_recall_at_{k}", 0) for k in [5, 10, 15] for c in configs]
ax.bar([f"{c}\n@5" for c in configs], [rq1_results[c].get("aggregate", {}).get("retrieval_recall_at_5", 0) for c in configs])
ax.set_ylabel('Recall@5')
ax.set_title('Recall@5')
ax.set_ylim([0, 1])

ax = axes[0, 1]
ax.bar([c[:15] for c in configs], [rq1_results[c].get("aggregate", {}).get("retrieval_recall_at_10", 0) for c in configs])
ax.set_ylabel('Recall@10')
ax.set_title('Recall@10')
ax.set_ylim([0, 1])

ax = axes[0, 2]
ax.bar([c[:15] for c in configs], [rq1_results[c].get("aggregate", {}).get("retrieval_recall_at_15", 0) for c in configs])
ax.set_ylabel('Recall@15')
ax.set_title('Recall@15')
ax.set_ylim([0, 1])

# Answer quality metrics
ax = axes[1, 0]
ax.bar([c[:15] for c in configs], [rq1_results[c].get("aggregate", {}).get("retrieval_mrr", 0) for c in configs])
ax.set_ylabel('MRR')
ax.set_title('Mean Reciprocal Rank')
ax.set_ylim([0, 1])

ax = axes[1, 1]
ax.bar([c[:15] for c in configs], [rq1_results[c].get("aggregate", {}).get("answer_token_f1", 0) for c in configs])
ax.set_ylabel('Token F1')
ax.set_title('Answer Token F1')
ax.set_ylim([0, 1])

ax = axes[1, 2]
ax.bar([c[:15] for c in configs], [rq1_results[c].get("aggregate", {}).get("retrieval_coverage", 0) for c in configs])
ax.set_ylabel('Coverage')
ax.set_title('Evidence Coverage')
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'rq1_comparison.png', dpi=150, bbox_inches='tight')
print("Visualization saved to:", ARTIFACTS_DIR / 'rq1_comparison.png')
plt.show()

## Key Findings

In [ ]:
# Analyze which chunking strategy wins on each metric
print("\n" + "="*60)
print("RQ1 SUMMARY: Chunking Strategy Impact")
print("="*60)

metric_keys = [
    ("retrieval_recall_at_5", "Recall@5"),
    ("retrieval_recall_at_10", "Recall@10"),
    ("retrieval_recall_at_15", "Recall@15"),
    ("retrieval_precision_at_5", "Precision@5"),
    ("retrieval_precision_at_10", "Precision@10"),
    ("retrieval_precision_at_15", "Precision@15"),
    ("retrieval_mrr", "MRR"),
    ("retrieval_coverage", "Coverage"),
    ("answer_token_f1", "Token F1"),
    ("answer_citation_f1", "Citation F1"),
]

for metric_key, metric_name in metric_keys:
    values = {name: rq1_results[name].get("aggregate", {}).get(metric_key, 0) 
              for name in rq1_results.keys()}
    winner = max(values.items(), key=lambda x: x[1])
    print(f"\n{metric_name}:")
    for name, value in sorted(values.items()):
        marker = " <- BEST" if name == winner[0] else ""
        print(f"  {name:30s}: {value:.4f}{marker}")

In [ ]:
# Save RQ1 results to disk
eval_results_dir = ARTIFACTS_DIR / "evaluation_results"
eval_results_dir.mkdir(parents=True, exist_ok=True)

output_path = eval_results_dir / "rq1_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(rq1_results, f, ensure_ascii=False, indent=2)

print(f"\nRQ1 results saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")